# 07 — Selected Model: Final Evaluation and Report Artifacts

**MarineGuard AI | Original YOLO26n checkpoint**

This notebook independently reruns **evaluation only** for the selected original checkpoint. It does not train, tune, or modify the model or the source dataset. It saves complete original-validation and TrashCan diagnostic metrics, per-class results, and Ultralytics evaluation figures to a separate output directory.

**Interpretation:** The original validation split contains only seven Net instances and is not a source-independent test of Net generalization. TrashCan was revisited during development and should be described as an **external-domain diagnostic**, not a pristine final test set.

## 1. Environment and experiment paths

Use the recorded Ultralytics version. Save all evaluation outputs separately from model-selection runs.

In [ ]:
%pip install -q "ultralytics==8.4.216" "pandas>=2.2,<3" "matplotlib>=3.8" pyyaml

from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
import ultralytics
import yaml
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')

DATASET = Path('/content/drive/MyDrive/MarineGuard_YOLO/MarineGuard_YOLO_Fixed')
CHECKPOINT = Path(
    '/content/drive/MyDrive/MarineGuard_Model_Selection/'
    'budget20_30_v1/runs_budget20_30_v1/final/yolo26n/weights/best.pt'
)
REPORT = Path('/content/drive/MyDrive/MarineGuard_Model_Selection/final_model_reporting_v1')
REPORT.mkdir(parents=True, exist_ok=True)
DATA_YAML = DATASET / 'data.yaml'

assert CHECKPOINT.is_file(), f'Original checkpoint not found: {CHECKPOINT}'
assert DATA_YAML.is_file(), f'Dataset configuration not found: {DATA_YAML}'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

print('Checkpoint:', CHECKPOINT)
print('Data configuration:', DATA_YAML)
print('Evaluation device:', DEVICE)
print('Output:', REPORT)
print('Ultralytics:', ultralytics.__version__)

## 2. Dataset and checkpoint verification

Confirm class order, expected split sizes, and target annotations before reporting any model metrics. This is a structural check, not a new annotation-quality audit.

In [ ]:
config = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
raw_names = config['names']
class_names = [raw_names.get(i, raw_names.get(str(i))) for i in (0, 1)] if isinstance(raw_names, dict) else list(raw_names)
assert [str(x).lower() for x in class_names] == ['net', 'rope'], class_names

model = YOLO(str(CHECKPOINT))
actual_names = {int(k): str(v).lower() for k, v in model.names.items()}
assert actual_names == {0: 'net', 1: 'rope'}, actual_names

splits = {'val': 178, 'test': 214}
for split, expected_images in splits.items():
    img_dir = DATASET / 'images' / split
    lbl_dir = DATASET / 'labels' / split
    assert img_dir.is_dir() and lbl_dir.is_dir(), (img_dir, lbl_dir)
    image_paths = [p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    assert len(image_paths) == expected_images, f'{split}: expected {expected_images}, found {len(image_paths)}'
    print(f'{split}: {len(image_paths)} images')

print('Class mapping verified:', model.names)

## 3. Original validation and external-domain diagnostic

Ultralytics calculates COCO-style mAP50 and mAP50–95, mean precision/recall, and per-class metrics. `conf=0.001` is used for the full detection metric curves; it is **not** the class-specific Streamlit display threshold. The validation and external datasets should be interpreted separately.

In [ ]:
def evaluate_split(split: str, label: str):
    metrics = model.val(
        data=str(DATA_YAML),
        split=split,
        imgsz=640,
        batch=4,
        device=DEVICE,
        workers=0,
        conf=0.001,
        plots=True,
        project=str(REPORT / 'ultralytics'),
        name=label,
        exist_ok=True,
        verbose=False,
    )
    overall = {
        'dataset': label, 'split': split,
        'mAP50': float(metrics.box.map50),
        'mAP50_95': float(metrics.box.map),
        'mean_precision': float(metrics.box.mp),
        'mean_recall': float(metrics.box.mr),
    }
    per_class = []
    for pos, class_id in enumerate(metrics.box.ap_class_index):
        cid = int(class_id)
        per_class.append({
            'dataset': label, 'class': model.names[cid],
            'precision': float(metrics.box.p[pos]),
            'recall': float(metrics.box.r[pos]),
            'mAP50': float(metrics.box.ap50[pos]),
            'mAP50_95': float(metrics.box.maps[cid]),
        })
    return overall, per_class

all_metrics = []
class_metrics = []
for split, label in [('val', 'original_validation'), ('test', 'trashcan_external_diagnostic')]:
    overall, classes = evaluate_split(split, label)
    all_metrics.append(overall)
    class_metrics.extend(classes)
    print(f'Completed: {label}')

overall_df = pd.DataFrame(all_metrics)
class_df = pd.DataFrame(class_metrics)
overall_df.to_csv(REPORT / 'overall_metrics.csv', index=False)
class_df.to_csv(REPORT / 'class_metrics.csv', index=False)
display(overall_df.round(5))
display(class_df.round(5))
print('Saved overall and class metrics to:', REPORT)

## 4. Export report figures

Create figures from the just-evaluated checkpoint, without manually entering performance numbers. Ultralytics also saves confusion matrices and PR/F1 curves within the output folder when plotting is supported.

In [ ]:
def format_ax(ax, title, ylabel):
    ax.set_title(title, fontweight='bold', pad=15)
    ax.set_ylabel(ylabel)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.15)
    ax.set_axisbelow(True)

def save_figure(filename):
    plt.gcf().savefig(REPORT / filename, dpi=220, bbox_inches='tight', facecolor='white')
    plt.close()

fig, ax = plt.subplots(figsize=(8.5, 5))
class_val = class_df[class_df['dataset'] == 'original_validation'].set_index('class')
classes = [model.names[0], model.names[1]]
values = [float(class_val.loc[name, 'mAP50_95']) * 100 for name in classes]
rect = ax.bar(classes, values, width=0.55)
ax.bar_label(rect, fmt='%.2f%%', padding=4)
ax.set_ylim(0, max(45, max(values) * 1.25))
format_ax(ax, 'Original YOLO26n · Validation mAP50–95 by class', 'mAP50–95 (%)')
save_figure('original_val_class_map.png')

fig, ax = plt.subplots(figsize=(8.5, 5))
values = overall_df['mAP50_95'].to_numpy() * 100
rect = ax.bar(['Original validation', 'TrashCan diagnostic'], values)
ax.bar_label(rect, labels=[f'{v:.3f}%' for v in values], padding=5)
ax.set_ylim(0, max(43, max(values) * 1.2))
format_ax(ax, 'Internal vs external detection performance', 'mAP50–95 (%)')
save_figure('internal_vs_external_map.png')

print('PNG figures saved in:', REPORT)
print('Additional plots from Ultralytics in:', REPORT / 'ultralytics')

## 5. Reproducibility and scope record

Record the exact checkpoint, environment and datasets used. No acceptance target or claimed reviewer time savings is introduced retrospectively.

In [ ]:
metadata = {
    'model': 'original YOLO26n, selected checkpoint',
    'checkpoint': str(CHECKPOINT),
    'dataset_yaml': str(DATA_YAML),
    'dataset_uses': {
        'val': 'Original 178-image SeaClear-derived validation; only 7 Net annotations',
        'test': '214-image TrashCan external-domain diagnostic; previously analyzed during development',
    },
    'inference_image_size': 640,
    'validation_confidence': 0.001,
    'application_class_thresholds': {'Net': 0.25, 'Rope': 0.15},
    'ultralytics_version': ultralytics.__version__,
    'python_version': platform.python_version(),
    'torch_version': torch.__version__,
    'device': str(DEVICE),
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'note': 'Evaluation only; no training, threshold tuning, or new data split',
}
(REPORT / 'evaluation_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved:', REPORT / 'evaluation_metadata.json')

## 6. Remaining application-level evaluation

**Not claimed as completed:** direct comparison against human underwater-image review, manual reviewer time, deployment reliability, and diverse-location generalization. These are separate evaluation activities. The recorded CPU benchmark in Notebook 01 measures model inference timing, not end-to-end human-review time saved. The application should not report a time-saving percentage until paired manual and model-assisted review measurements are available.

No training is needed to execute this notebook. To test Streamlit itself, use representative uploads and verify class labels, images with no detections, image orientation, response time, and downloading annotated results.